### test lobid models

In [59]:
import sys
import os
import json

# Fix Python path for notebook context
sys.path.insert(0, os.path.abspath(".."))

from src.enrichapi.models.lobidGND import DataLobidGND

# Simulated JSON payload from https://lobid.org/gnd/118610465.json
sample_data = {
    "type": ["Person", "AuthorityRecord"],
    "preferredName": "Schubert, Franz",
    "biographicalOrHistoricalInformation": ["Österr. Komponist"],
    "depiction": [
        {"id": "https://commons.wikimedia.org/wiki/Special:FilePath/Franz_Schubert.jpg"}
    ],
    "professionOrOccupation": [{"id": "https://...", "label": "Komponist"}],
    "dateOfBirth": ["1797-01-31"],
    "dateOfDeath": ["1828-11-19"],
    "geographicAreaCode": [
        {"id": "https://d-nb.info/standards/vocab/gnd/geographic-area-code#XA-AT", "label": "Österreich"}
    ]
}

# Validate directly into DataLobidGND
result = DataLobidGND(gndId="118610465", gndInformation=sample_data)

print(f"Parsed Model Type: {type(result.gndInformation).__name__}")
print(f"Name: {result.gndInformation.preferredName}")
print(f"Professions: {result.gndInformation.professionsOrOccupations}")
print(f"Depictions: {result.gndInformation.depictionURLs}")

# Test the geographic area code extraction
if result.gndInformation.geographicAreaCodes:
    geo = result.gndInformation.geographicAreaCodes[0]
    print(f"Geo Code: {geo.code}")      # XA-AT
    print(f"Geo URL: {geo.idURL}")      # https://d-nb.info/...#XA-AT

Parsed Model Type: PersonLobidGND
Name: Schubert, Franz
Professions: ['Komponist']
Depictions: ['https://commons.wikimedia.org/wiki/Special:FilePath/Franz_Schubert.jpg']
Geo Code: XA-AT
Geo URL: https://d-nb.info/standards/vocab/gnd/geographic-area-code#XA-AT


### new tests

#### Full Deep Extraction (SRU + Lobid + Wikidata Lookup) mit maxRecs implemented

In [66]:
# with Volltext-URL: AC17288271
# with abstract: AC03987726
# with inhaltsverzeichnis: AC07527091
# Journal/Newspaper: AC02908223
# Kant: +Z168276302
# Safranski Schopenhauer: AC01514953

import requests

url = "http://127.0.0.1:8000/enrich"

payload = {
    "iType": "bib",
    "institution": {
        "iName": "oenb",
        "identifier": "+Z168276302",
        "identifierType": "barcode",
        "fetchMarc21MD": True,
        "fetchLobidGND": True,
        "fetchSimilarByAuthor": True,
        "fetchSimilarBySubject": True,
        "fetchSimilarByClassification": True,
        "maxRecs": 10
    }
}

res = requests.post(url, json=payload)
res.raise_for_status()

res.json()

{'response': {'iType': 'bib',
  'result': {'identifier': '+Z168276302',
   'basicMarc21MD': {'title': {'titleMain': 'Grundlegung zur Metaphysik der Sitten',
     'titleRemainder': None,
     'titlePartNumber': [],
     'titlePartName': []},
    'mainEntry': {'name': 'Kant, Immanuel',
     'nameType': 'personal',
     'relator': ['aut'],
     'gndIdentifier': '118559796'},
    'addedEntries': [],
    'languageCodes': [],
    'languageCodesOriginal': [],
    'publicationCountryCodes': [],
    'edition': None,
    'physicalDescriptions': [],
    'publicationNotices': [{'pnType': 'other',
      'dating': None,
      'places': ['Riga'],
      'names': ['Hartknoch'],
      'dates': ['1785']}],
    'genreForms': [],
    'subjectHeadings': ['Ethik', 'Metaphysik'],
    'classifications': [],
    'bibMaterialType': 'Book (BK)',
    'bibResourceType': 'Book - Physical',
    'fullTextURLs': ['http://data.onb.ac.at/ABO/%2BZ168276302',
     'http://data.onb.ac.at/ABO/%2BZ201309403'],
    'abstracts'

In [45]:
import requests

url = "http://127.0.0.1:8000/enrich"

payload = {
    "iType": "bib",
    "institution": {
        "iName": "oenb",
        "identifier": "AC08958802",
        "identifierType": "ac",
        "fetchMarc21MD": True,
        "fetchSimilarByAuthor": True,
        "maxRecs": 10
    }
}

res = requests.post(url, json=payload)
res.raise_for_status()

res.json()

{'response': {'iType': 'bib',
  'result': {'identifier': 'AC08958802',
   'basicMarc21MD': {'title': {'titleMain': 'Gold',
     'titleRemainder': '[Gold in der Kunst, von der Antike bis zur Moderne ; dieser Katalog erscheint anlässlich der Ausstellung ... vom 15. März bis 17. Juni 2012 im Belvedere, Wien]',
     'titlePartNumber': [],
     'titlePartName': []},
    'mainEntry': None,
    'addedEntries': [{'name': 'Husslein-Arco, Agnes',
      'nameType': 'personal',
      'relator': ['edt'],
      'gndIdentifier': '13045978X'},
     {'name': 'Zaunschirm, Thomas',
      'nameType': 'personal',
      'relator': ['edt'],
      'gndIdentifier': '120560631'},
     {'name': 'Österreichische Galerie Belvedere',
      'nameType': 'corporate',
      'relator': ['ctb'],
      'gndIdentifier': '2165327-6'}],
    'languageCodes': ['ger'],
    'languageCodesOriginal': [],
    'publicationCountryCodes': ['XA-AT', 'XA-DE'],
    'edition': None,
    'physicalDescriptions': [{'extent': '367 S.',
      

#### Full Deep Extraction (SRU + Lobid + Wikidata Lookup)

In [50]:
import requests

url = "http://127.0.0.1:8000/enrich"

payload = {
    "iType": "bib",
    "institution": {
        "iName": "oenb",
        "identifier": "+Z168276302",
        "identifierType": "barcode",
        "fetchMarc21MD": True,
        "fetchLobidGND": True,
        "fetchWikidata": True
    }
}

res = requests.post(url, json=payload)
res.raise_for_status()

res.json()

{'response': {'iType': 'bib',
  'result': {'identifier': '+Z168276302',
   'basicMarc21MD': {'title': {'titleMain': 'Grundlegung zur Metaphysik der Sitten',
     'titleRemainder': None,
     'titlePartNumber': [],
     'titlePartName': []},
    'mainEntry': {'name': 'Kant, Immanuel',
     'nameType': 'personal',
     'relator': ['aut'],
     'gndIdentifier': '118559796'},
    'addedEntries': [],
    'languageCodes': [],
    'languageCodesOriginal': [],
    'publicationCountryCodes': [],
    'edition': None,
    'physicalDescriptions': [],
    'publicationNotices': [{'pnType': 'other',
      'dating': None,
      'places': ['Riga'],
      'names': ['Hartknoch'],
      'dates': ['1785']}],
    'genreForms': [],
    'subjectHeadings': ['Ethik', 'Metaphysik'],
    'classifications': [],
    'bibMaterialType': 'Book (BK)',
    'bibResourceType': 'Book - Physical',
    'fullTextURLs': ['http://data.onb.ac.at/ABO/%2BZ168276302',
     'http://data.onb.ac.at/ABO/%2BZ201309403'],
    'abstracts'

#### Selective Bypass Strategy

In [12]:
import requests

url = "http://127.0.0.1:8000/enrich"

payload = {
    "iType": "bib",
    "institution": {
        "iName": "oenb",
        "identifier": "AC12345678",
        "identifierType": "ac",
        "gndId": "118516477",
        "fetchMarc21MD": False,
        "fetchLobidGND": True,
        "fetchWikidata": False
    }
}

res = requests.post(url, json=payload)
res.raise_for_status()

res.json()

{'response': {'iType': 'bib',
  'result': {'identifier': 'AC12345678',
   'basicMarc21MD': None,
   'additionalRecsSRU': None,
   'gndInfoLobid': {'gndId': '118516477',
    'gndInformation': {'entityTypes': ['Person', 'AuthorityRecord'],
     'depictionURLs': [],
     'geographicAreaCodes': [],
     'sameAs': [],
     'homepage': [],
     'gndSubjectCategories': [],
     'preferredName': 'Albert Einstein',
     'biographicalOrHistoricalInformation': [],
     'gndType': 'person'}},
   'wikidataData': None,
   'bookCover': None,
   'bookDescription': None,
   'iName': 'oenb',
   'identifierType': 'ac'}}}

#### Use additionalRecs functionality with mock data

In [44]:
import requests

url = "http://127.0.0.1:8000/enrich"

# test payload validating the concurrent subsidiary SRU routing
payload = {
    "iType": "bib",
    "institution": {
        "iName": "oenb",
        "identifier": "Z168276302",
        "identifierType": "barcode",
        "fetchMarc21MD": True,
        "fetchLobidGND": True,
        "fetchWikidata": True,
        
        # query paths
        "fetchSimilarByAuthor": True,
        "fetchSimilarBySubject": True,
        "fetchSimilarByClassification": True
    }
}

try:
    print("Sending enrichment request...")
    res = requests.post(url, json=payload)
    res.raise_for_status()
    
    data = res.json()
    print("Success! Response received structure:")
    
    # inspect sub-model key inside the JSON response
    result_node = data["response"]["result"]
    print(f"Identifier: {result_node['identifier']}")
    print(f"Basic MARC21 Found: {result_node['basicMarc21MD'] is not None}")
    print(f"Lobid Data Found: {result_node['gndInfoLobid'] is not None}")
    print(f"Wikidata Data Found: {result_node['wikidataData'] is not None}")
    
    # print newly wired multi-strategy array
    print("\n--- Additional Records SRU Output ---")
    print(result_node["additionalRecsSRU"])

except requests.exceptions.RequestException as e:
    print(f"Request failed: {e}")
    if e.response is not None:
        print(f"Server response: {e.response.text}")

Sending enrichment request...
Success! Response received structure:
Identifier: Z168276302
Basic MARC21 Found: True
Lobid Data Found: True
Wikidata Data Found: True

--- Additional Records SRU Output ---
{'records': [{'searchType': 'classification', 'classifications': [], 'maxRecs': 5, 'additionalRecs': []}, {'searchType': 'author', 'name': 'Kant, Immanuel', 'maxRecs': 5, 'additionalRecs': [{'ac': 'AC14036130', 'titleMain': '[Ohne Titel]', 'callNumbers': ['Autogr. 144/82-1'], 'isbns': [], 'issns': []}, {'ac': 'AC10005051', 'titleMain': 'A kedély hatalmáról, miszerint a puszta feltett szándék által uralkodhatni a kóros érzelmeken', 'callNumbers': ['58579-B'], 'isbns': [], 'issns': []}, {'ac': 'AC13902513', 'titleMain': 'Adorno und Kant', 'callNumbers': ['Cod. Ser. n. 44243'], 'isbns': [], 'issns': []}, {'ac': 'AC13904422', 'titleMain': 'Adorno und Kant', 'callNumbers': ['Cod. Ser. n. 44204'], 'isbns': [], 'issns': []}, {'ac': 'AC04210882', 'titleMain': 'Al eterna paco', 'callNumbers': [

### old tests

In [1]:
# try a request for v2 -> needs to be post request

import requests

url = "http://127.0.0.1:8000/enrich"

# payload matches nested EnrichmentRequest structure:
# Level 3: iType ("bib")
# Level 2: institution
# Level 1: iName ("oenb") + barcode
payload = {
    "iType": "bib",
    "institution": {
        "iName": "oenb",
        "barcode": "Z168276302"
    }
}

res = requests.post(url, json=payload)
res.raise_for_status()

res.json()
    

{'iType': 'bib',
 'result': {'iName': 'oenb',
  'barcode': 'Z168276302',
  'metadata': {'title245': 'Grundlegung zur Metaphysik der Sitten',
   'author100': 'Kant, Immanuel',
   'gndID': '(DE-588)118559796',
   'titlesGND': ['Gesammelte Schriften',
    'Die Religion innerhalb der Grenzen der bloßen Vernunft',
    'Beobachtungen über das Gefühl des Schönen und Erhabenen',
    'Metaphysische Anfangsgründe der Naturwissenschaft',
    'Kritik der Urteilskraft']}}}

In [ ]:
# try a request for v1

import requests

# omit "+" in barcode

# Schopenhauer, Die Welt als Wille und Vorstellung
#barcode = "Z17057080X"
#callNum = "19.V.9"
ac = "AC10298143"

# Kant, GMS
#barcode = "Z168276302"
#callNum = "219886-B"
#ac = "AC10004994"

#url = f"http://127.0.0.1:8000/barcode/{barcode}"
#url = f"http://127.0.0.1:8000/callNumber/{callNum}"
url = f"http://127.0.0.1:8000/acNumber/{ac}"

res = requests.get(url)

res.json()

{'id': 'AC10298143',
 'idType': 'acNumber',
 'title245': 'Die Welt als Wille und Vorstellung; 4 Bücher, nebst einem Anh., der die Kritik der Kantischen Philosophie enthält.',
 'author100': 'Schopenhauer, Arthur',
 'gndID': '(DE-588)118610465',
 'titlesGND': ['Über das Sehen und die Farben',
  '"Kan Menneskets frie Villie bevises af dets Selvbevidsthed?"',
  'Die beiden Grundprobleme der Ethik, behandelt in zwei akadem. Preisschriften',
  'Über die vierfache Wurzel des Satzes vom hinreichenden Grunde',
  'Über die Lebensalter',
  'Aphorismen zur Lebensweisheit',
  'Über die Weiber',
  'Die Welt als Wille und Vorstellung']}

In [14]:
import requests

# gnd id
gndID = "118610465"

url = f"https://lobid.org/gnd/{gndID}.json"

res = requests.get(url)

resultDict = res.json()

resultDict["publication"]

['Über das Sehen und die Farben',
 '"Kan Menneskets frie Villie bevises af dets Selvbevidsthed?"',
 'Die beiden Grundprobleme der Ethik, behandelt in zwei akadem. Preisschriften',
 'Über die vierfache Wurzel des Satzes vom hinreichenden Grunde',
 'Über die Lebensalter',
 'Aphorismen zur Lebensweisheit',
 'Über die Weiber',
 'Die Welt als Wille und Vorstellung']